In [3]:
# Cell 1: Imports and interactive widget setup
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
from ipywidgets import interact, IntRangeSlider, IntSlider
%matplotlib inline

In [10]:
def detect_bell(image_path, h_range, s_range, v_range, min_area_pct):
    """
    Processes a single image to locate the gold bell based on HSV thresholds.
    """
    # 1. Load the image and convert color spaces
    # Note: cv2 reads as BGR, but Pi Camera captures as RGB. Adjust if loading raw arrays.
    bgr_img = cv2.imread(image_path)
    if bgr_img is None:
        print(f"Error: Could not load image from {image_path}")
        return
        
    rgb_img = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2RGB)
    hsv_img = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2HSV)
    
    # 2. Define the lower and upper HSV limits from sliders
    lower_gold = np.array([h_range[0], s_range[0], v_range[0]])
    upper_gold = np.array([h_range[1], s_range[1], v_range[1]])
    
    # 3. Create the binary mask (isolated gold color)
    mask = cv2.inRange(hsv_img, lower_gold, upper_gold)
    
    # Clean up small noise/specks using morphological opening
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    
    # 4. Count matching pixels and calculate area percentages
    total_pixels = mask.shape[0] * mask.shape[1]
    gold_pixels = cv2.countNonZero(mask)
    detected_area_pct = (gold_pixels / total_pixels) * 100
    
    # 5. Determine if the bell is officially "detected"
    is_detected = detected_area_pct >= min_area_pct
    status_text = f"BELL DETECTED ({detected_area_pct:.1f}% of frame)" if is_detected else f"Searching... ({detected_area_pct:.1f}% of frame)"
    status_color = (0, 255, 0) if is_detected else (255, 0, 0) # Green if true, Red if false
    
    # 6. Draw results for visual tuning
    output_img = rgb_img.copy()
    
    # Optional: Find contours to draw a bounding box around the blurry blob
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        # Find the largest matching blob
        largest_contour = max(contours, key=cv2.contourArea)
        if cv2.contourArea(largest_contour) > 500: # ignore tiny noise
            x, y, w, h = cv2.boundingRect(largest_contour)
            cv2.rectangle(output_img, (x, y), (x + w, y + h), status_color, 4)
            
    # Add a text overlay to the output image
    cv2.putText(output_img, status_text, (15, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, status_color, 3)

    # 7. Plot Side-by-Side inside Notebook
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(output_img)
    axes[0].set_title("Detection Output")
    axes[0].axis('off')
    
    axes[1].imshow(mask, cmap='gray')
    axes[1].set_title("Color Mask (What the Robot Sees)")
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

# =====================================================================
# Cell 2: Run the Interactive Tuning Environment
# =====================================================================

from ipywidgets import interact, IntRangeSlider, IntSlider, fixed # <-- Added fixed here
from pathlib import Path

# IMG_FOLDER = Path("../data/extracted_frames/may24/bell_indoors")     
IMG_FOLDER = Path("../data/extracted_frames/may15/bell1")   
IMG_PATH = IMG_FOLDER / "frame_000000.jpg"     

# Convert the Path object to a standard string just to keep OpenCV perfectly happy
str_image_path = str(IMG_PATH)

print("Adjust sliders to isolate the gold color profile while rejecting background elements:")
interact(
    detect_bell,
    image_path=fixed(str_image_path), # <-- Wrap it in fixed() so ipywidgets ignores it
    h_range=IntRangeSlider(value=[15, 30], min=0, max=180, step=1, description="Hue Range (H)"),
    s_range=IntRangeSlider(value=[80, 255], min=0, max=255, step=1, description="Sat Range (S)"),
    v_range=IntRangeSlider(value=[80, 255], min=0, max=255, step=1, description="Val Range (V)"),
    min_area_pct=IntSlider(value=25, min=1, max=100, step=1, description="Trigger Area %")
);

Adjust sliders to isolate the gold color profile while rejecting background elements:


interactive(children=(IntRangeSlider(value=(15, 30), description='Hue Range (H)', max=180), IntRangeSlider(val…